In [1]:
import polars as pl
import pandas as pd
import xxhash
import catboost as cb
import numpy as np
from catboost import CatBoostRanker

In [2]:
DATA_PATH = "/home/jupyter/filestore/storage/datasets/train_dataset.parquet"

In [2]:
dataset = pl.read_parquet("/home/jupyter/filestore/storage/datasets/train_dataset_with_feats.parquet")

In [3]:
user_segment = pl.read_parquet("/home/jupyter/project/user_segment.parquet")

In [4]:
oos_dataset = (
    dataset
    .with_columns(
        pl.col("user_id").apply(lambda x: abs(xxhash.xxh64(str(x), seed=42).intdigest()) % 100).alias("user_segment")
    )
    .filter(pl.col("user_segment").is_in(list(range(2))))
    .drop("user_segment")
    .sort("user_id")
)
            
train_dataset = (
    dataset
    .with_columns(
        pl.col("user_id").apply(lambda x: abs(xxhash.xxh64(str(x), seed=42).intdigest()) % 100).alias("user_segment")
    )
    .filter(pl.col("user_segment").is_in(list(range(2, 10))))
    .drop("user_segment")
    .sort("user_id")
)

In [5]:
cat_features = train_dataset.select(pl.col(pl.Utf8)).columns
features = [col_name for col_name in train_dataset.columns if col_name not in ["user_id", "item_id", "target"]]

In [6]:
test_df = (
    oos_dataset
    .to_pandas()
)

test_df[cat_features] = test_df[cat_features].fillna("MISSING")

test_pool = cb.Pool(
    test_df[features],
    cat_features=cat_features
)

In [34]:
#scores = pl.read_parquet("/home/jupyter/project/all_candidates.parquet")

In [6]:
cb_params = {
    'iterations': 200,
    'learning_rate': 0.5,
    'border_count': 254,
    'boosting_type': 'Plain',
    'early_stopping_rounds': 5,
    'task_type': "GPU",
    'depth': 4,
    'loss_function': 'QuerySoftMax',
    'bootstrap_type': 'No',
    'random_state': 42,
    'eval_metric': 'PrecisionAt:top=1'
}

In [7]:
train_df = (
    train_dataset
    .to_pandas()
    .sample(frac=1)
    .sort_values(by="user_id")
    .replace({None: np.nan})
)
train_label = train_df["target"]
train_group_id = train_df["user_id"]
train_df[cat_features] = train_df[cat_features].fillna("MISSING")

In [6]:
oos_df = (
    oos_dataset
    .to_pandas()
    .sample(frac=1)
    .sort_values(by="user_id")
    .replace({None: np.nan})
)
oos_label = oos_df["target"]
oos_group_id = oos_df["user_id"]
oos_df[cat_features] = oos_df[cat_features].fillna("MISSING")

In [9]:
train_pool = cb.Pool(
    train_df[features],
    label=train_label,
    cat_features=cat_features,
    group_id=train_group_id
)

In [7]:
test_pool = cb.Pool(
    oos_df[features],
    label=oos_label,
    cat_features=cat_features,
    group_id=oos_group_id
)

In [100]:
feature_importance = model.get_feature_importance(train_pool)
feature_names = features

feature_importances = sorted([(feat_name, feat_importance) for feat_name, feat_importance in zip(feature_names, feature_importance)], key=lambda x: -x[1])

In [ ]:
feature_importances[:50]

In [18]:
cv_results, cv_models = cb.cv(
    train_pool,
    cb_params,
    fold_count=5,
    return_models=True,
    partition_random_seed=42
)

Training on fold [0/5]


Default metric period is 5 because PrecisionAt is/are not implemented for GPU
Metric PrecisionAt:top=1 is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


0:	learn: 0.3091078	test: 0.3135408	best: 0.3135408 (0)	total: 141ms	remaining: 28.1s
1:	total: 272ms	remaining: 26.9s
2:	total: 398ms	remaining: 26.1s
3:	total: 522ms	remaining: 25.6s
4:	total: 648ms	remaining: 25.3s
5:	learn: 0.5862522	test: 0.5805215	best: 0.5805215 (5)	total: 774ms	remaining: 25s
6:	total: 898ms	remaining: 24.8s
7:	total: 1.02s	remaining: 24.5s
8:	total: 1.15s	remaining: 24.4s
9:	total: 1.27s	remaining: 24.2s
10:	learn: 0.6341290	test: 0.6328878	best: 0.6328878 (10)	total: 1.41s	remaining: 24.2s
11:	total: 1.53s	remaining: 24s
12:	total: 1.66s	remaining: 23.9s
13:	total: 1.79s	remaining: 23.7s
14:	total: 1.92s	remaining: 23.7s
15:	learn: 0.6592842	test: 0.6543602	best: 0.6543602 (15)	total: 2.05s	remaining: 23.6s
16:	total: 2.18s	remaining: 23.4s
17:	total: 2.3s	remaining: 23.2s
18:	total: 2.43s	remaining: 23.1s
19:	total: 2.56s	remaining: 23s
20:	learn: 0.6663659	test: 0.6595092	best: 0.6615907 (19)	total: 2.69s	remaining: 23s
21:	total: 2.82s	remaining: 22.8s
22:

Metric PrecisionAt:top=1 is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


0:	learn: 0.3103306	test: 0.3100683	best: 0.3100683 (0)	total: 143ms	remaining: 28.4s
1:	total: 271ms	remaining: 26.8s
2:	total: 399ms	remaining: 26.2s
3:	total: 525ms	remaining: 25.7s
4:	total: 653ms	remaining: 25.4s
5:	learn: 0.5795087	test: 0.5784112	best: 0.5784112 (5)	total: 786ms	remaining: 25.4s
6:	total: 917ms	remaining: 25.3s
7:	total: 1.05s	remaining: 25.2s
8:	total: 1.18s	remaining: 25s
9:	total: 1.31s	remaining: 24.9s
10:	learn: 0.6314635	test: 0.6321665	best: 0.6321665 (10)	total: 1.45s	remaining: 24.9s
11:	total: 1.58s	remaining: 24.8s
12:	total: 1.71s	remaining: 24.6s
13:	total: 1.84s	remaining: 24.5s
14:	total: 1.97s	remaining: 24.3s
15:	learn: 0.6611559	test: 0.6654384	best: 0.6654384 (15)	total: 2.12s	remaining: 24.3s
16:	total: 2.25s	remaining: 24.2s
17:	total: 2.37s	remaining: 24s
18:	total: 2.5s	remaining: 23.8s
19:	total: 2.62s	remaining: 23.6s
20:	learn: 0.6692438	test: 0.6747589	best: 0.6756259 (19)	total: 2.75s	remaining: 23.4s
21:	total: 2.87s	remaining: 23.3s

...

0:	learn: 0.3114287	test: 0.3022116	best: 0.3022116 (0)	total: 144ms	remaining: 28.7s
1:	total: 273ms	remaining: 27s
2:	total: 399ms	remaining: 26.2s
3:	total: 525ms	remaining: 25.7s
4:	total: 650ms	remaining: 25.3s
5:	learn: 0.5862768	test: 0.5831790	best: 0.5831790 (5)	total: 778ms	remaining: 25.2s
6:	total: 910ms	remaining: 25.1s
7:	total: 1.03s	remaining: 24.8s
8:	total: 1.16s	remaining: 24.6s
9:	total: 1.28s	remaining: 24.4s
10:	learn: 0.6349598	test: 0.6306787	best: 0.6306787 (10)	total: 1.41s	remaining: 24.3s
11:	total: 1.54s	remaining: 24.1s
12:	total: 1.67s	remaining: 24s
13:	total: 1.79s	remaining: 23.8s
14:	total: 1.91s	remaining: 23.6s
15:	learn: 0.6589179	test: 0.6553001	best: 0.6553001 (15)	total: 2.05s	remaining: 23.5s
16:	total: 2.17s	remaining: 23.4s
17:	total: 2.3s	remaining: 23.3s
18:	total: 2.42s	remaining: 23.1s
19:	total: 2.55s	remaining: 22.9s
20:	learn: 0.6697881	test: 0.6672840	best: 0.6672840 (20)	total: 2.68s	remaining: 22.8s
21:	total: 2.81s	remaining: 22.7s

Metric PrecisionAt:top=1 is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


0:	learn: 0.3090984	test: 0.3150203	best: 0.3150203 (0)	total: 141ms	remaining: 28.1s
1:	total: 271ms	remaining: 26.9s
2:	total: 397ms	remaining: 26.1s
3:	total: 521ms	remaining: 25.5s
4:	total: 649ms	remaining: 25.3s
5:	learn: 0.5821311	test: 0.5933414	best: 0.5933414 (5)	total: 778ms	remaining: 25.2s
6:	total: 904ms	remaining: 24.9s
7:	total: 1.03s	remaining: 24.7s
8:	total: 1.16s	remaining: 24.5s
9:	total: 1.28s	remaining: 24.3s
10:	learn: 0.6333880	test: 0.6409186	best: 0.6409186 (10)	total: 1.41s	remaining: 24.3s
11:	total: 1.54s	remaining: 24.2s
12:	total: 1.67s	remaining: 24s
13:	total: 1.79s	remaining: 23.8s
14:	total: 1.91s	remaining: 23.5s
15:	learn: 0.6595082	test: 0.6613559	best: 0.6613559 (15)	total: 2.04s	remaining: 23.4s
16:	total: 2.17s	remaining: 23.3s
17:	total: 2.29s	remaining: 23.1s
18:	total: 2.41s	remaining: 23s
19:	total: 2.54s	remaining: 22.9s
20:	learn: 0.6720219	test: 0.6728931	best: 0.6728931 (20)	total: 2.68s	remaining: 22.8s
21:	total: 2.8s	remaining: 22.7s

Metric PrecisionAt:top=1 is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


0:	learn: 0.3098676	test: 0.3084050	best: 0.3084050 (0)	total: 141ms	remaining: 28.1s
1:	total: 272ms	remaining: 26.9s
2:	total: 397ms	remaining: 26s
3:	total: 519ms	remaining: 25.4s
4:	total: 643ms	remaining: 25.1s
5:	learn: 0.5856694	test: 0.5838297	best: 0.5838297 (5)	total: 770ms	remaining: 24.9s
6:	total: 893ms	remaining: 24.6s
7:	total: 1.01s	remaining: 24.4s
8:	total: 1.14s	remaining: 24.2s
9:	total: 1.26s	remaining: 24s
10:	learn: 0.6317456	test: 0.6268476	best: 0.6268476 (10)	total: 1.39s	remaining: 23.9s
11:	total: 1.51s	remaining: 23.7s
12:	total: 1.64s	remaining: 23.6s
13:	total: 1.76s	remaining: 23.4s
14:	total: 1.89s	remaining: 23.3s
15:	learn: 0.6603248	test: 0.6554158	best: 0.6554158 (15)	total: 2.02s	remaining: 23.2s
16:	total: 2.14s	remaining: 23s
17:	total: 2.27s	remaining: 22.9s
18:	total: 2.39s	remaining: 22.8s
19:	total: 2.52s	remaining: 22.7s
20:	learn: 0.6748738	test: 0.6724024	best: 0.6724024 (20)	total: 2.65s	remaining: 22.6s
21:	total: 2.77s	remaining: 22.4s


In [20]:
model = cb.sum_models(cv_models, weights=[1.0 / len(cv_models)] * len(cv_models))

In [24]:
model.save_model("ranker_v1.cbm")

In [25]:
model.save_model("/home/jupyter/filestore/storage/models/ranker_v1.cbm")

In [8]:
new_model = CatBoostRanker()
new_model.load_model('ranker_v1.cbm')

CatBoostRanker()

In [15]:
train_metric = new_model.eval_metrics(train_pool, 'PrecisionAt:top=10')
test_metric = new_model.eval_metrics(test_pool, 'PrecisionAt:top=10')

In [16]:
train_metric["PrecisionAt:top=10"][-1]

0.24599305423138226

In [17]:
test_metric["PrecisionAt:top=10"][-1]

0.24050808660737844

In [22]:
train_metric["PrecisionAt:top=1"][-1]

0.7205531607623465

In [23]:
test_metric["PrecisionAt:top=1"][-1]

0.7135102005078364

In [12]:
recall_10 = new_model.eval_metrics(test_pool, 'RecallAt:top=10')
recall_1 = new_model.eval_metrics(test_pool, 'RecallAt:top=1')
recall_1["RecallAt:top=1"][-1], recall_10["RecallAt:top=10"][-1]

(0.38981970409205197, 0.8726766347096637)